In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# -----------------------------
# Paths
# -----------------------------
BASE = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data")
PROFILE_DIR = BASE / "processed" / "census_profiles"
META_DIR = BASE / "raw" / "tabular" / "Metadata"
CORR_DIR = BASE / "raw" / "tabular" / "asgs_correspondences"
OUTPUT_DIR = BASE / "processed" / "census_lookup_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SA2_PROFILE = PROFILE_DIR / "census_sa2_profile.csv"
SAL_PROFILE = PROFILE_DIR / "census_sal_profile.csv"
POA_PROFILE = PROFILE_DIR / "census_poa_profile.csv"

META_FILES = list(META_DIR.glob("*.xlsx")) + list(META_DIR.glob("*.csv"))


# -----------------------------
# Helpers
# -----------------------------
def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

def normalize_columns(df):
    out = df.copy()
    out.columns = [norm_col(c) for c in out.columns]
    return out

def save_csv(df, name):
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    return path

def build_lookup_from_profile(profile_path, code_col_name, name_col_name=None):
    df = pd.read_csv(profile_path, low_memory=False)
    df = normalize_columns(df)

    if code_col_name not in df.columns:
        raise ValueError(f"{code_col_name} not found in {profile_path.name}. Found: {df.columns[:10].tolist()} ...")

    out = pd.DataFrame()
    out[code_col_name] = df[code_col_name].astype(str).str.strip()

    if name_col_name and name_col_name in df.columns:
        out[name_col_name] = df[name_col_name].astype(str).str.strip()

    out = out.dropna().drop_duplicates()
    return out

def build_poa_lookup_from_profile():
    df = pd.read_csv(POA_PROFILE, low_memory=False)
    df = normalize_columns(df)

    if "poa_code_2021" not in df.columns:
        raise ValueError(f"poa_code_2021 not found in {POA_PROFILE.name}. First columns: {df.columns[:10].tolist()}")

    poa = pd.DataFrame()
    poa["poa_code_2021"] = df["poa_code_2021"].astype(str).str.strip()
    poa = poa.drop_duplicates()

    poa["postcode"] = poa["poa_code_2021"].str.extract(r"(\d+)")[0]
    poa["display_label"] = "Postcode " + poa["postcode"].astype(str)

    return poa


# -----------------------------
# Metadata search
# -----------------------------
def find_best_code_name_table_in_excel(target_prefix, meta_files):
    target_prefix = target_prefix.lower()

    for file in meta_files:
        if file.suffix.lower() not in [".xlsx", ".xls"]:
            continue

        try:
            xls = pd.ExcelFile(file)
        except Exception:
            continue

        for sheet in xls.sheet_names:
            try:
                sample = pd.read_excel(file, sheet_name=sheet, nrows=50)
            except Exception:
                continue

            sample = normalize_columns(sample)
            cols = set(sample.columns)

            code_candidates = [c for c in cols if target_prefix in c and "code" in c]
            name_candidates = [c for c in cols if target_prefix in c and "name" in c]

            if code_candidates and name_candidates:
                try:
                    full = pd.read_excel(file, sheet_name=sheet)
                    full = normalize_columns(full)

                    code_col = code_candidates[0]
                    name_col = name_candidates[0]

                    out = full[[code_col, name_col]].copy()
                    out = out.rename(columns={
                        code_col: f"{target_prefix}_code_2021",
                        name_col: f"{target_prefix}_name_2021"
                    })
                    out = out.dropna().drop_duplicates()

                    if len(out) > 0:
                        print(f"Found {target_prefix.upper()} lookup in {file.name} -> sheet '{sheet}'")
                        return out
                except Exception:
                    continue

    return None

def find_best_code_name_table_in_csv(target_prefix, meta_files):
    target_prefix = target_prefix.lower()

    for file in meta_files:
        if file.suffix.lower() != ".csv":
            continue

        try:
            df = pd.read_csv(file, low_memory=False)
        except Exception:
            continue

        df = normalize_columns(df)
        cols = set(df.columns)

        code_candidates = [c for c in cols if target_prefix in c and "code" in c]
        name_candidates = [c for c in cols if target_prefix in c and "name" in c]

        if code_candidates and name_candidates:
            code_col = code_candidates[0]
            name_col = name_candidates[0]

            out = df[[code_col, name_col]].copy()
            out = out.rename(columns={
                code_col: f"{target_prefix}_code_2021",
                name_col: f"{target_prefix}_name_2021"
            })
            out = out.dropna().drop_duplicates()

            if len(out) > 0:
                print(f"Found {target_prefix.upper()} lookup in {file.name}")
                return out

    return None

def build_lookup_from_metadata(target_prefix):
    df = find_best_code_name_table_in_excel(target_prefix, META_FILES)
    if df is not None:
        return normalize_columns(df)

    df = find_best_code_name_table_in_csv(target_prefix, META_FILES)
    if df is not None:
        return normalize_columns(df)

    return None


# -----------------------------
# Build lookup tables
# -----------------------------
sa2_lookup = build_lookup_from_metadata("sa2")
sal_lookup = build_lookup_from_metadata("sal")
poa_lookup = build_lookup_from_metadata("poa")

if sa2_lookup is not None:
    sa2_lookup = sa2_lookup[["sa2_code_2021", "sa2_name_2021"]].drop_duplicates()
    save_csv(sa2_lookup, "sa2_lookup.csv")
else:
    print("Could not build sa2_lookup from local metadata.")

if sal_lookup is not None:
    sal_lookup = sal_lookup[["sal_code_2021", "sal_name_2021"]].drop_duplicates()
    save_csv(sal_lookup, "sal_lookup.csv")
else:
    print("Could not build sal_lookup from local metadata.")

if poa_lookup is None:
    print("No POA name metadata found locally; building POA lookup from POA profile codes")
    poa_lookup = build_poa_lookup_from_profile()

poa_lookup = normalize_columns(poa_lookup)
poa_lookup = poa_lookup[["poa_code_2021", "postcode", "display_label"]].drop_duplicates()
save_csv(poa_lookup, "poa_lookup.csv")


Could not build sa2_lookup from local metadata.
Could not build sal_lookup from local metadata.
No POA name metadata found locally; building POA lookup from POA profile codes
Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_lookup_tables\poa_lookup.csv


WindowsPath('C:/Users/AmanRoy/Documents/GitHub/NeighbourWise/Data/processed/census_lookup_tables/poa_lookup.csv')

In [3]:
from pathlib import Path
import pandas as pd
import re

META_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\Metadata")
meta_files = list(META_DIR.glob("*.xlsx")) + list(META_DIR.glob("*.xls"))

def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

for file in meta_files:
    print("\n" + "=" * 100)
    print(f"FILE: {file.name}")
    print("=" * 100)

    try:
        xls = pd.ExcelFile(file)
    except Exception as e:
        print(f"Could not open {file.name}: {e}")
        continue

    print("Sheets:")
    for sheet in xls.sheet_names:
        print(" -", sheet)

    print("\nSheet column preview:")
    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(file, sheet_name=sheet, nrows=5)
            cols = [norm_col(c) for c in df.columns]

            hits = [c for c in cols if any(k in c for k in ["sa2", "sal", "poa", "code", "name"])]
            if hits:
                print(f"\nSheet: {sheet}")
                print("Matching columns:", hits[:30])
        except Exception as e:
            print(f"\nSheet: {sheet}")
            print(f"Could not read: {e}")



FILE: 2021Census_geog_desc_1st_2nd_3rd_release.xlsx
Sheets:
 - 2021_ASGS_MAIN_Structures
 - 2021_ASGS_GCCSA_Structure
 - 2021_ASGS_Indigenous_Structures
 - 2021_ASGS_SOS_UCL_Structures
 - 2021_ASGS_SUA_Structures
 - 2021_ASGS_Non_ABS_Structures
 - 2021_ASGS_RA_Structures

Sheet column preview:

Sheet: 2021_ASGS_MAIN_Structures
Matching columns: ['census_code_2021', 'agss_code_2021', 'census_name_2021']

Sheet: 2021_ASGS_GCCSA_Structure
Matching columns: ['census_code_2021', 'agss_code_2021', 'census_name_2021']

Sheet: 2021_ASGS_Indigenous_Structures
Matching columns: ['census_code_2021', 'agss_code_2021', 'census_name_2021']

Sheet: 2021_ASGS_SOS_UCL_Structures
Matching columns: ['census_code_2021', 'agss_code_2021', 'census_name_2021']

Sheet: 2021_ASGS_SUA_Structures
Matching columns: ['census_code_2021', 'agss_code_2021', 'census_name_2021']

Sheet: 2021_ASGS_Non_ABS_Structures
Matching columns: ['census_code_2021', 'agss_code_2021', 'census_name_2021']

Sheet: 2021_ASGS_RA_Struct

In [4]:
from pathlib import Path
import pandas as pd
import re

META_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\Metadata")
meta_files = list(META_DIR.glob("*.xlsx")) + list(META_DIR.glob("*.xls"))

def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

for file in meta_files:
    try:
        xls = pd.ExcelFile(file)
    except:
        continue

    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(file, sheet_name=sheet, nrows=10)
            df.columns = [norm_col(c) for c in df.columns]
            cols = df.columns.tolist()

            if any("sa2" in c for c in cols) or any("sal" in c for c in cols) or any("poa" in c for c in cols):
                print("\n" + "=" * 100)
                print(f"FILE: {file.name}")
                print(f"SHEET: {sheet}")
                print("COLUMNS:")
                print(cols[:50])
                display(df.head(5))
        except:
            continue


In [5]:
from pathlib import Path
import pandas as pd
import re

file = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\Metadata\2021Census_geog_desc_1st_2nd_3rd_release.xlsx")

def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

sheet = "2021_ASGS_MAIN_Structures"

df = pd.read_excel(file, sheet_name=sheet)
df.columns = [norm_col(c) for c in df.columns]

print("Columns:")
print(df.columns.tolist())

print("\nShape:")
print(df.shape)

print("\nFirst 10 rows:")
display(df.head(10))

print("\nUnique values preview per column:")
for col in df.columns:
    vals = df[col].dropna().astype(str).unique()[:20]
    print(f"\n{col}:")
    print(vals)


Columns:
['asgs_structure', 'census_code_2021', 'agss_code_2021', 'census_name_2021', 'area_sqkm']

Shape:
(64791, 5)

First 10 rows:


,asgs_structure,census_code_2021,agss_code_2021,census_name_2021,area_sqkm
0,AUS,AUS,AUS,AUSTRALIA,7.688095e+06
1,STE,1,1,New South Wales,8.007977e+05
2,STE,2,2,Victoria,2.274962e+05
3,STE,3,3,Queensland,1.730171e+06
4,STE,4,4,South Australia,9.842314e+05
5,STE,5,5,Western Australia,2.526632e+06
6,STE,6,6,Tasmania,6.801754e+04
7,STE,7,7,Northern Territory,1.348134e+06
8,STE,8,8,Australian Capital Territory,2.358133e+03
9,STE,9,9,Other Territories,2.557420e+02



Unique values preview per column:

asgs_structure:
['AUS' 'STE' 'SA4' 'SA3' 'SA2' 'SA1']

census_code_2021:
['AUS' '1' '2' '3' '4' '5' '6' '7' '8' '9' '101' '102' '103' '104' '105'
 '106' '107' '108' '109' '110']

agss_code_2021:
['AUS' '1' '2' '3' '4' '5' '6' '7' '8' '9' '101' '102' '103' '104' '105'
 '106' '107' '108' '109' '110']

census_name_2021:
['AUSTRALIA' 'New South Wales' 'Victoria' 'Queensland' 'South Australia'
 'Western Australia' 'Tasmania' 'Northern Territory'
 'Australian Capital Territory' 'Other Territories' 'Capital Region'
 'Central Coast' 'Central West' 'Coffs Harbour - Grafton'
 'Far West and Orana' 'Hunter Valley exc Newcastle' 'Illawarra'
 'Mid North Coast' 'Murray' 'New England and North West']

area_sqkm:
['7688094.938' '800797.6591' '227496.2479' '1730171.221' '984231.4202'
 '2526632.483' '68017.5393' '1348134.493' '2358.133' '255.742'
 '51896.2444' '1681.0088' '70297.06' '13229.7577' '339355.6461'
 '21491.2917' '1539.2405' '18851.4986' '97796.4898' '99139.8

In [6]:
for col in df.columns:
    series = df[col].astype(str).str.upper()
    mask = (
        series.str.contains("SA2", na=False) |
        series.str.contains("SAL", na=False) |
        series.str.contains("POA", na=False)
    )
    if mask.any():
        print("\n" + "="*80)
        print("COLUMN:", col)
        display(df.loc[mask, [col]].drop_duplicates().head(30))



COLUMN: asgs_structure


,asgs_structure
475,SA2



COLUMN: census_name_2021


,census_name_2021
373,Salisbury
813,Hassall Grove - Plumpton
1228,Sale
1713,Salisbury - Nathan
1822,Crows Nest - Rosalie
1934,Brassall
2237,Salisbury East
2238,Salisbury North
2485,Hocking - Pearsall


In [7]:
mask = df.astype(str).apply(
    lambda s: s.str.upper().str.contains("SA2|SAL|POA", na=False)
).any(axis=1)

preview = df.loc[mask].copy()

print("Matching rows:", len(preview))
display(preview.head(30))


Matching rows: 2473


,asgs_structure,census_code_2021,agss_code_2021,census_name_2021,area_sqkm
373,SA3,40204,40204,Salisbury,150.1709
475,SA2,101021007,101021007,Braidwood,3418.3525
476,SA2,101021008,101021008,Karabar,6.9825
477,SA2,101021009,101021009,Queanbeyan,4.7620
478,SA2,101021010,101021010,Queanbeyan - East,13.0032
479,SA2,101021012,101021012,Queanbeyan West - Jerrabomberra,13.6748
480,SA2,101021610,101021610,Googong,6.9164
481,SA2,101021611,101021611,Queanbeyan Surrounds,3047.7057
482,SA2,101031013,101031013,Bombala,3990.1440
483,SA2,101031014,101031014,Cooma,103.6371


In [8]:
from pathlib import Path
import pandas as pd
import re

META_FILE = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\Metadata\2021Census_geog_desc_1st_2nd_3rd_release.xlsx")
OUTPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_lookup_tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

# Load relevant sheets
main_df = pd.read_excel(META_FILE, sheet_name="2021_ASGS_MAIN_Structures")
main_df.columns = [norm_col(c) for c in main_df.columns]

non_abs_df = pd.read_excel(META_FILE, sheet_name="2021_ASGS_Non_ABS_Structures")
non_abs_df.columns = [norm_col(c) for c in non_abs_df.columns]

print("MAIN unique structures:")
print(sorted(main_df["asgs_structure"].dropna().astype(str).unique().tolist()))

print("\nNON-ABS unique structures:")
print(sorted(non_abs_df["asgs_structure"].dropna().astype(str).unique().tolist()))


MAIN unique structures:
['AUS', 'SA1', 'SA2', 'SA3', 'SA4', 'STE']

NON-ABS unique structures:
['AUS', 'CED', 'LGA', 'POA', 'SAL', 'SED']


In [9]:
sa2_lookup = (
    main_df.loc[main_df["asgs_structure"].astype(str).str.upper() == "SA2",
                ["census_code_2021", "census_name_2021"]]
    .copy()
    .rename(columns={
        "census_code_2021": "sa2_code_2021",
        "census_name_2021": "sa2_name_2021"
    })
    .dropna()
    .drop_duplicates()
)

sa2_lookup["sa2_code_2021"] = sa2_lookup["sa2_code_2021"].astype(str).str.strip()
sa2_lookup["sa2_name_2021"] = sa2_lookup["sa2_name_2021"].astype(str).str.strip()

sa2_lookup.to_csv(OUTPUT_DIR / "sa2_lookup.csv", index=False)

print(sa2_lookup.shape)
display(sa2_lookup.head(10))
print(f"Saved: {OUTPUT_DIR / 'sa2_lookup.csv'}")


(2472, 2)


,sa2_code_2021,sa2_name_2021
475,101021007,Braidwood
476,101021008,Karabar
477,101021009,Queanbeyan
478,101021010,Queanbeyan - East
479,101021012,Queanbeyan West - Jerrabomberra
480,101021610,Googong
481,101021611,Queanbeyan Surrounds
482,101031013,Bombala
483,101031014,Cooma
484,101031015,Cooma Surrounds


Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_lookup_tables\sa2_lookup.csv


In [10]:
for structure in ["SAL", "POA"]:
    subset = non_abs_df.loc[
        non_abs_df["asgs_structure"].astype(str).str.upper() == structure
    ].copy()

    print(f"\n{structure} rows: {len(subset)}")
    if len(subset) > 0:
        display(subset.head(10))



SAL rows: 15352


,asgs_structure,census_code_2021,agss_code_2021,census_name_2021,area_sqkm
3829,SAL,SAL10001,10001,Aarons Pass,82.7639
3830,SAL,SAL10002,10002,Abbotsbury,4.9788
3831,SAL,SAL10003,10003,Abbotsford (NSW),1.0180
3832,SAL,SAL10004,10004,Abercrombie,2.9775
3833,SAL,SAL10005,10005,Abercrombie River,127.1701
3834,SAL,SAL10006,10006,Aberdare,3.2961
3835,SAL,SAL10007,10007,Aberdeen (NSW),41.6327
3836,SAL,SAL10008,10008,Aberfoyle,340.4792
3837,SAL,SAL10009,10009,Aberglasslyn,6.2154
3838,SAL,SAL10010,10010,Abermain,15.3822



POA rows: 2643


,asgs_structure,census_code_2021,agss_code_2021,census_name_2021,area_sqkm
1186,POA,POA2448,2448,"2448, NSW",115.0676
1187,POA,POA2000,2000,"2000, NSW",4.2871
1188,POA,POA2007,2007,"2007, NSW",0.5588
1189,POA,POA2008,2008,"2008, NSW",0.8450
1190,POA,POA2009,2009,"2009, NSW",0.9299
1191,POA,POA2010,2010,"2010, NSW",2.1733
1192,POA,POA2011,2011,"2011, NSW",1.4054
1193,POA,POA2015,2015,"2015, NSW",4.2157
1194,POA,POA2016,2016,"2016, NSW",1.1717
1195,POA,POA2017,2017,"2017, NSW",1.9386


In [11]:
sal_lookup = (
    non_abs_df.loc[non_abs_df["asgs_structure"].astype(str).str.upper() == "SAL",
                   ["census_code_2021", "census_name_2021"]]
    .copy()
    .rename(columns={
        "census_code_2021": "sal_code_2021",
        "census_name_2021": "sal_name_2021"
    })
    .dropna()
    .drop_duplicates()
)

sal_lookup["sal_code_2021"] = sal_lookup["sal_code_2021"].astype(str).str.strip()
sal_lookup["sal_name_2021"] = sal_lookup["sal_name_2021"].astype(str).str.strip()

sal_lookup.to_csv(OUTPUT_DIR / "sal_lookup.csv", index=False)

print(sal_lookup.shape)
display(sal_lookup.head(10))
print(f"Saved: {OUTPUT_DIR / 'sal_lookup.csv'}")


poa_lookup = (
    non_abs_df.loc[non_abs_df["asgs_structure"].astype(str).str.upper() == "POA",
                   ["census_code_2021", "census_name_2021"]]
    .copy()
    .rename(columns={
        "census_code_2021": "poa_code_2021",
        "census_name_2021": "display_label"
    })
    .dropna()
    .drop_duplicates()
)

poa_lookup["poa_code_2021"] = poa_lookup["poa_code_2021"].astype(str).str.strip()
poa_lookup["postcode"] = poa_lookup["poa_code_2021"].astype(str).str.extract(r"(\d+)")[0]

# If display_label is not useful, overwrite with a simple label
poa_lookup["display_label"] = "Postcode " + poa_lookup["postcode"].astype(str)

poa_lookup = poa_lookup[["poa_code_2021", "postcode", "display_label"]]

poa_lookup.to_csv(OUTPUT_DIR / "poa_lookup.csv", index=False)

print(poa_lookup.shape)
display(poa_lookup.head(10))
print(f"Saved: {OUTPUT_DIR / 'poa_lookup.csv'}")


(15352, 2)


,sal_code_2021,sal_name_2021
3829,SAL10001,Aarons Pass
3830,SAL10002,Abbotsbury
3831,SAL10003,Abbotsford (NSW)
3832,SAL10004,Abercrombie
3833,SAL10005,Abercrombie River
3834,SAL10006,Aberdare
3835,SAL10007,Aberdeen (NSW)
3836,SAL10008,Aberfoyle
3837,SAL10009,Aberglasslyn
3838,SAL10010,Abermain


Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_lookup_tables\sal_lookup.csv
(2643, 3)


,poa_code_2021,postcode,display_label
1186,POA2448,2448,Postcode 2448
1187,POA2000,2000,Postcode 2000
1188,POA2007,2007,Postcode 2007
1189,POA2008,2008,Postcode 2008
1190,POA2009,2009,Postcode 2009
1191,POA2010,2010,Postcode 2010
1192,POA2011,2011,Postcode 2011
1193,POA2015,2015,Postcode 2015
1194,POA2016,2016,Postcode 2016
1195,POA2017,2017,Postcode 2017


Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_lookup_tables\poa_lookup.csv


In [12]:
from pathlib import Path
import pandas as pd
import re

BASE = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data")
LOOKUP_DIR = BASE / "processed" / "census_lookup_tables"
PROFILE_DIR = BASE / "processed" / "census_profiles"

def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

# Load profile tables
sa2_profile = pd.read_csv(PROFILE_DIR / "census_sa2_profile.csv", low_memory=False)
sal_profile = pd.read_csv(PROFILE_DIR / "census_sal_profile.csv", low_memory=False)
poa_profile = pd.read_csv(PROFILE_DIR / "census_poa_profile.csv", low_memory=False)

sa2_profile.columns = [norm_col(c) for c in sa2_profile.columns]
sal_profile.columns = [norm_col(c) for c in sal_profile.columns]
poa_profile.columns = [norm_col(c) for c in poa_profile.columns]

# Load lookup tables you already created
sa2_lookup = pd.read_csv(LOOKUP_DIR / "sa2_lookup.csv", low_memory=False)
sal_lookup = pd.read_csv(LOOKUP_DIR / "sal_lookup.csv", low_memory=False)
poa_lookup = pd.read_csv(LOOKUP_DIR / "poa_lookup.csv", low_memory=False)

sa2_lookup.columns = [norm_col(c) for c in sa2_lookup.columns]
sal_lookup.columns = [norm_col(c) for c in sal_lookup.columns]
poa_lookup.columns = [norm_col(c) for c in poa_lookup.columns]

# Force key columns to string
sa2_profile["sa2_code_2021"] = sa2_profile["sa2_code_2021"].astype(str).str.strip()
sal_profile["sal_code_2021"] = sal_profile["sal_code_2021"].astype(str).str.strip()
poa_profile["poa_code_2021"] = poa_profile["poa_code_2021"].astype(str).str.strip()

sa2_lookup["sa2_code_2021"] = sa2_lookup["sa2_code_2021"].astype(str).str.strip()
sal_lookup["sal_code_2021"] = sal_lookup["sal_code_2021"].astype(str).str.strip()
poa_lookup["poa_code_2021"] = poa_lookup["poa_code_2021"].astype(str).str.strip()

# Filter lookups to VIC-only profile codes
sa2_lookup_vic = sa2_lookup[
    sa2_lookup["sa2_code_2021"].isin(sa2_profile["sa2_code_2021"])
].drop_duplicates().sort_values("sa2_code_2021")

sal_lookup_vic = sal_lookup[
    sal_lookup["sal_code_2021"].isin(sal_profile["sal_code_2021"])
].drop_duplicates().sort_values("sal_code_2021")

poa_lookup_vic = poa_lookup[
    poa_lookup["poa_code_2021"].isin(poa_profile["poa_code_2021"])
].drop_duplicates().sort_values("poa_code_2021")

# Save final VIC-only lookup tables
sa2_lookup_vic.to_csv(LOOKUP_DIR / "sa2_lookup_vic.csv", index=False)
sal_lookup_vic.to_csv(LOOKUP_DIR / "sal_lookup_vic.csv", index=False)
poa_lookup_vic.to_csv(LOOKUP_DIR / "poa_lookup_vic.csv", index=False)

print("Final filtered shapes:")
print("sa2_lookup_vic:", sa2_lookup_vic.shape)
print("sal_lookup_vic:", sal_lookup_vic.shape)
print("poa_lookup_vic:", poa_lookup_vic.shape)

display(sa2_lookup_vic.head())
display(sal_lookup_vic.head())
display(poa_lookup_vic.head())


Final filtered shapes:
sa2_lookup_vic: (524, 2)
sal_lookup_vic: (2946, 2)
poa_lookup_vic: (694, 3)


,sa2_code_2021,sa2_name_2021
644,201011001,Alfredton
645,201011002,Ballarat
646,201011005,Buninyong
647,201011006,Delacombe
648,201011007,Smythes Creek


,sal_code_2021,sal_name_2021
4544,SAL20001,Abbeyard
4545,SAL20002,Abbotsford (Vic.)
4546,SAL20003,Aberfeldie
4547,SAL20004,Aberfeldy
4548,SAL20005,Acheron


,poa_code_2021,postcode,display_label
1301,POA3000,3000,Postcode 3000
1302,POA3002,3002,Postcode 3002
612,POA3003,3003,Postcode 3003
613,POA3004,3004,Postcode 3004
614,POA3006,3006,Postcode 3006


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data")
RAW_DIR = BASE / "raw" / "tabular"
LOOKUP_DIR = BASE / "processed" / "census_lookup_tables"
OUT_DIR = BASE / "processed" / "census_crosswalks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POA_TO_SA2_FILE = RAW_DIR / "CG_POA_2021_SA2_2021.xlsx"
SAL_TO_POA_FILE = RAW_DIR / "CG_SAL_2021_POA_2021.xlsx"

SA2_LOOKUP_FILE = LOOKUP_DIR / "sa2_lookup_vic.csv"
SAL_LOOKUP_FILE = LOOKUP_DIR / "sal_lookup_vic.csv"
POA_LOOKUP_FILE = LOOKUP_DIR / "poa_lookup_vic.csv"


def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")


def normalize_columns(df):
    out = df.copy()
    out.columns = [norm_col(c) for c in out.columns]
    return out


def load_excel_any_sheet(path):
    xls = pd.ExcelFile(path)
    best_df = None
    best_score = -1
    best_sheet = None

    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(path, sheet_name=sheet)
            df = normalize_columns(df)

            # Prefer sheets with code columns
            score = sum(
                any(k in c for k in ["poa", "sa2", "sal", "ratio", "code"])
                for c in df.columns
            )

            if score > best_score and len(df) > 0:
                best_score = score
                best_df = df
                best_sheet = sheet
        except Exception:
            continue

    print(f"Using sheet '{best_sheet}' from {path.name}")
    print("Columns:", best_df.columns.tolist())
    return best_df


def find_col(cols, must_have=None, can_have=None):
    must_have = must_have or []
    can_have = can_have or []

    for c in cols:
        if all(x in c for x in must_have) and all((y in c) for y in can_have):
            return c

    for c in cols:
        if all(x in c for x in must_have):
            return c

    return None


def confidence_from_pct(series):
    return np.select(
        [
            series >= 90,
            series >= 60
        ],
        ["high", "medium"],
        default="low"
    )


# -------------------------------------------------
# Load VIC lookup tables
# -------------------------------------------------
sa2_lookup = pd.read_csv(SA2_LOOKUP_FILE, low_memory=False)
sal_lookup = pd.read_csv(SAL_LOOKUP_FILE, low_memory=False)
poa_lookup = pd.read_csv(POA_LOOKUP_FILE, low_memory=False)

sa2_lookup = normalize_columns(sa2_lookup)
sal_lookup = normalize_columns(sal_lookup)
poa_lookup = normalize_columns(poa_lookup)

sa2_lookup["sa2_code_2021"] = sa2_lookup["sa2_code_2021"].astype(str).str.strip()
sal_lookup["sal_code_2021"] = sal_lookup["sal_code_2021"].astype(str).str.strip()
poa_lookup["poa_code_2021"] = poa_lookup["poa_code_2021"].astype(str).str.strip()


# -------------------------------------------------
# 1. Build POA -> SA2
# -------------------------------------------------
poa_sa2_raw = load_excel_any_sheet(POA_TO_SA2_FILE)
cols = poa_sa2_raw.columns.tolist()

poa_col = find_col(cols, must_have=["poa", "2021"])
sa2_col = find_col(cols, must_have=["sa2", "2021"])
ratio_col = find_col(cols, must_have=["ratio"]) or find_col(cols, must_have=["proportion"]) or find_col(cols, must_have=["pct"])

if poa_col is None or sa2_col is None:
    raise ValueError(f"Could not identify POA/SA2 columns in {POA_TO_SA2_FILE.name}")

poa_to_sa2 = poa_sa2_raw[[c for c in [poa_col, sa2_col, ratio_col] if c is not None]].copy()
poa_to_sa2 = poa_to_sa2.rename(columns={
    poa_col: "poa_code_2021",
    sa2_col: "sa2_code_2021",
    **({ratio_col: "overlap_area_pct"} if ratio_col else {})
})

poa_to_sa2["poa_code_2021"] = poa_to_sa2["poa_code_2021"].astype(str).str.strip()
poa_to_sa2["sa2_code_2021"] = poa_to_sa2["sa2_code_2021"].astype(str).str.strip()

# Keep only VIC rows
poa_to_sa2 = poa_to_sa2[
    poa_to_sa2["poa_code_2021"].isin(poa_lookup["poa_code_2021"]) &
    poa_to_sa2["sa2_code_2021"].isin(sa2_lookup["sa2_code_2021"])
].copy()

if "overlap_area_pct" in poa_to_sa2.columns:
    poa_to_sa2["overlap_area_pct"] = pd.to_numeric(poa_to_sa2["overlap_area_pct"], errors="coerce")

    # Convert 0-1 ratios to percentages if needed
    if poa_to_sa2["overlap_area_pct"].dropna().max() <= 1:
        poa_to_sa2["overlap_area_pct"] = (poa_to_sa2["overlap_area_pct"] * 100).round(2)
    else:
        poa_to_sa2["overlap_area_pct"] = poa_to_sa2["overlap_area_pct"].round(2)
else:
    poa_to_sa2["overlap_area_pct"] = np.nan

poa_to_sa2 = poa_to_sa2.merge(
    poa_lookup[["poa_code_2021", "postcode"]],
    on="poa_code_2021",
    how="left"
)

poa_to_sa2 = poa_to_sa2.merge(
    sa2_lookup[["sa2_code_2021", "sa2_name_2021"]],
    on="sa2_code_2021",
    how="left"
)

poa_to_sa2["confidence"] = confidence_from_pct(poa_to_sa2["overlap_area_pct"].fillna(0))

poa_to_sa2 = poa_to_sa2[
    ["poa_code_2021", "postcode", "sa2_code_2021", "sa2_name_2021", "overlap_area_pct", "confidence"]
].drop_duplicates()

poa_to_sa2 = poa_to_sa2.sort_values(
    ["poa_code_2021", "overlap_area_pct"],
    ascending=[True, False]
)

poa_to_sa2.to_csv(OUT_DIR / "poa_to_sa2_vic.csv", index=False)
print("Saved:", OUT_DIR / "poa_to_sa2_vic.csv")
print("poa_to_sa2_vic shape:", poa_to_sa2.shape)
display(poa_to_sa2.head(10))


# -------------------------------------------------
# 2. Build SAL -> POA
# -------------------------------------------------
sal_poa_raw = load_excel_any_sheet(SAL_TO_POA_FILE)
cols = sal_poa_raw.columns.tolist()

sal_col = find_col(cols, must_have=["sal", "2021"])
poa_col_2 = find_col(cols, must_have=["poa", "2021"])
ratio_col_2 = find_col(cols, must_have=["ratio"]) or find_col(cols, must_have=["proportion"]) or find_col(cols, must_have=["pct"])

if sal_col is None or poa_col_2 is None:
    raise ValueError(f"Could not identify SAL/POA columns in {SAL_TO_POA_FILE.name}")

sal_to_poa = sal_poa_raw[[c for c in [sal_col, poa_col_2, ratio_col_2] if c is not None]].copy()
sal_to_poa = sal_to_poa.rename(columns={
    sal_col: "sal_code_2021",
    poa_col_2: "poa_code_2021",
    **({ratio_col_2: "sal_poa_overlap_pct"} if ratio_col_2 else {})
})

sal_to_poa["sal_code_2021"] = sal_to_poa["sal_code_2021"].astype(str).str.strip()
sal_to_poa["poa_code_2021"] = sal_to_poa["poa_code_2021"].astype(str).str.strip()

# Keep only VIC rows
sal_to_poa = sal_to_poa[
    sal_to_poa["sal_code_2021"].isin(sal_lookup["sal_code_2021"]) &
    sal_to_poa["poa_code_2021"].isin(poa_lookup["poa_code_2021"])
].copy()

if "sal_poa_overlap_pct" in sal_to_poa.columns:
    sal_to_poa["sal_poa_overlap_pct"] = pd.to_numeric(sal_to_poa["sal_poa_overlap_pct"], errors="coerce")
    if sal_to_poa["sal_poa_overlap_pct"].dropna().max() <= 1:
        sal_to_poa["sal_poa_overlap_pct"] = (sal_to_poa["sal_poa_overlap_pct"] * 100).round(2)
    else:
        sal_to_poa["sal_poa_overlap_pct"] = sal_to_poa["sal_poa_overlap_pct"].round(2)
else:
    sal_to_poa["sal_poa_overlap_pct"] = np.nan

print("sal_to_poa filtered shape:", sal_to_poa.shape)


# -------------------------------------------------
# 3. Derive SAL -> SA2 via POA
# -------------------------------------------------
sal_to_sa2 = sal_to_poa.merge(
    poa_to_sa2[["poa_code_2021", "sa2_code_2021", "overlap_area_pct"]],
    on="poa_code_2021",
    how="inner",
    suffixes=("", "_poa_sa2")
)

# Combined score: SAL->POA weight times POA->SA2 weight
sal_to_sa2["overlap_area_pct"] = (
    sal_to_sa2["sal_poa_overlap_pct"].fillna(0) *
    sal_to_sa2["overlap_area_pct"].fillna(0) / 100
).round(2)

# Aggregate to SAL-SA2 pairs
sal_to_sa2 = (
    sal_to_sa2.groupby(["sal_code_2021", "sa2_code_2021"], as_index=False)["overlap_area_pct"]
    .sum()
)

sal_to_sa2 = sal_to_sa2.merge(
    sal_lookup[["sal_code_2021", "sal_name_2021"]],
    on="sal_code_2021",
    how="left"
)

sal_to_sa2 = sal_to_sa2.merge(
    sa2_lookup[["sa2_code_2021", "sa2_name_2021"]],
    on="sa2_code_2021",
    how="left"
)

sal_to_sa2["confidence"] = confidence_from_pct(sal_to_sa2["overlap_area_pct"].fillna(0))

sal_to_sa2 = sal_to_sa2[
    ["sal_code_2021", "sal_name_2021", "sa2_code_2021", "sa2_name_2021", "overlap_area_pct", "confidence"]
].drop_duplicates()

sal_to_sa2 = sal_to_sa2.sort_values(
    ["sal_code_2021", "overlap_area_pct"],
    ascending=[True, False]
)

sal_to_sa2.to_csv(OUT_DIR / "sal_to_sa2_vic.csv", index=False)
print("Saved:", OUT_DIR / "sal_to_sa2_vic.csv")
print("sal_to_sa2_vic shape:", sal_to_sa2.shape)
display(sal_to_sa2.head(10))


Using sheet 'CG_POA_2021_SA2_2021_All' from CG_POA_2021_SA2_2021.xlsx
Columns: ['poa_code_2021', 'sa2_code_2021', 'sa2_name_2021', 'ratio_from_to', 'indiv_to_region_qlty_indicator', 'overall_quality_indicator', 'bmos_null_flag']
Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_crosswalks\poa_to_sa2_vic.csv
poa_to_sa2_vic shape: (0, 6)


,poa_code_2021,postcode,sa2_code_2021,sa2_name_2021,overlap_area_pct,confidence


Using sheet 'CG_2021_SAL_2021_POA_GRID21_All' from CG_SAL_2021_POA_2021.xlsx
Columns: ['sal_code_2021', 'sal_name_2021', 'poa_code_2021', 'ratio_from_to', 'indiv_to_region_qlty_indicator', 'overall_quality_indicator', 'bmos_null_flag']
sal_to_poa filtered shape: (0, 3)
Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_crosswalks\sal_to_sa2_vic.csv
sal_to_sa2_vic shape: (0, 6)


,sal_code_2021,sal_name_2021,sa2_code_2021,sa2_name_2021,overlap_area_pct,confidence


In [2]:
for name in ["poa_to_sa2_vic.csv", "sal_to_sa2_vic.csv"]:
    path = OUT_DIR / name
    df = pd.read_csv(path)
    print(f"\n{name}: {df.shape}")
    display(df.head(10))



poa_to_sa2_vic.csv: (0, 6)


,poa_code_2021,postcode,sa2_code_2021,sa2_name_2021,overlap_area_pct,confidence



sal_to_sa2_vic.csv: (0, 6)


,sal_code_2021,sal_name_2021,sa2_code_2021,sa2_name_2021,overlap_area_pct,confidence


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data")
RAW_DIR = BASE / "raw" / "tabular"
LOOKUP_DIR = BASE / "processed" / "census_lookup_tables"
OUT_DIR = BASE / "processed" / "census_crosswalks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POA_TO_SA2_FILE = RAW_DIR / "CG_POA_2021_SA2_2021.xlsx"
SAL_TO_POA_FILE = RAW_DIR / "CG_SAL_2021_POA_2021.xlsx"

def norm_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

def normalize_columns(df):
    out = df.copy()
    out.columns = [norm_col(c) for c in out.columns]
    return out

def norm_code(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    s = re.sub(r"\.0$", "", s)
    s = s.replace(" ", "")
    return s.upper()

def norm_poa(val):
    s = norm_code(val)
    if pd.isna(s):
        return s
    digits = re.sub(r"\D", "", s)
    return f"POA{digits}" if digits else s

def norm_sal(val):
    s = norm_code(val)
    if pd.isna(s):
        return s
    digits = re.sub(r"\D", "", s)
    return f"SAL{digits}" if digits else s

def norm_sa2(val):
    s = norm_code(val)
    if pd.isna(s):
        return s
    digits = re.sub(r"\D", "", s)
    return digits if digits else s

def confidence_from_pct(series):
    return np.select(
        [series >= 90, series >= 60],
        ["high", "medium"],
        default="low"
    )

# Load VIC lookups
poa_lookup = normalize_columns(pd.read_csv(LOOKUP_DIR / "poa_lookup_vic.csv", low_memory=False))
sal_lookup = normalize_columns(pd.read_csv(LOOKUP_DIR / "sal_lookup_vic.csv", low_memory=False))
sa2_lookup = normalize_columns(pd.read_csv(LOOKUP_DIR / "sa2_lookup_vic.csv", low_memory=False))

poa_lookup["poa_code_2021"] = poa_lookup["poa_code_2021"].map(norm_poa)
sal_lookup["sal_code_2021"] = sal_lookup["sal_code_2021"].map(norm_sal)
sa2_lookup["sa2_code_2021"] = sa2_lookup["sa2_code_2021"].map(norm_sa2)

# Load raw crosswalks
poa_raw = normalize_columns(pd.read_excel(POA_TO_SA2_FILE, sheet_name="CG_POA_2021_SA2_2021_All"))
sal_raw = normalize_columns(pd.read_excel(SAL_TO_POA_FILE, sheet_name="CG_2021_SAL_2021_POA_GRID21_All"))

poa_raw["poa_code_2021"] = poa_raw["poa_code_2021"].map(norm_poa)
poa_raw["sa2_code_2021"] = poa_raw["sa2_code_2021"].map(norm_sa2)

sal_raw["sal_code_2021"] = sal_raw["sal_code_2021"].map(norm_sal)
sal_raw["poa_code_2021"] = sal_raw["poa_code_2021"].map(norm_poa)

# POA -> SA2
poa_to_sa2 = poa_raw[["poa_code_2021", "sa2_code_2021", "ratio_from_to"]].copy()
poa_to_sa2 = poa_to_sa2.rename(columns={"ratio_from_to": "overlap_area_pct"})

poa_to_sa2["overlap_area_pct"] = pd.to_numeric(poa_to_sa2["overlap_area_pct"], errors="coerce")
if poa_to_sa2["overlap_area_pct"].dropna().max() <= 1:
    poa_to_sa2["overlap_area_pct"] = (poa_to_sa2["overlap_area_pct"] * 100).round(2)
else:
    poa_to_sa2["overlap_area_pct"] = poa_to_sa2["overlap_area_pct"].round(2)

poa_to_sa2 = poa_to_sa2[
    poa_to_sa2["poa_code_2021"].isin(poa_lookup["poa_code_2021"]) &
    poa_to_sa2["sa2_code_2021"].isin(sa2_lookup["sa2_code_2021"])
].copy()

poa_to_sa2 = poa_to_sa2.merge(
    poa_lookup[["poa_code_2021", "postcode"]],
    on="poa_code_2021",
    how="left"
)

poa_to_sa2 = poa_to_sa2.merge(
    sa2_lookup[["sa2_code_2021", "sa2_name_2021"]],
    on="sa2_code_2021",
    how="left"
)

poa_to_sa2["confidence"] = confidence_from_pct(poa_to_sa2["overlap_area_pct"].fillna(0))
poa_to_sa2 = poa_to_sa2[
    ["poa_code_2021", "postcode", "sa2_code_2021", "sa2_name_2021", "overlap_area_pct", "confidence"]
].drop_duplicates().sort_values(["poa_code_2021", "overlap_area_pct"], ascending=[True, False])

poa_to_sa2.to_csv(OUT_DIR / "poa_to_sa2_vic.csv", index=False)

print("poa_to_sa2_vic:", poa_to_sa2.shape)
display(poa_to_sa2.head(10))

# SAL -> POA
sal_to_poa = sal_raw[["sal_code_2021", "poa_code_2021", "ratio_from_to"]].copy()
sal_to_poa = sal_to_poa.rename(columns={"ratio_from_to": "sal_poa_overlap_pct"})

sal_to_poa["sal_poa_overlap_pct"] = pd.to_numeric(sal_to_poa["sal_poa_overlap_pct"], errors="coerce")
if sal_to_poa["sal_poa_overlap_pct"].dropna().max() <= 1:
    sal_to_poa["sal_poa_overlap_pct"] = (sal_to_poa["sal_poa_overlap_pct"] * 100).round(2)
else:
    sal_to_poa["sal_poa_overlap_pct"] = sal_to_poa["sal_poa_overlap_pct"].round(2)

sal_to_poa = sal_to_poa[
    sal_to_poa["sal_code_2021"].isin(sal_lookup["sal_code_2021"]) &
    sal_to_poa["poa_code_2021"].isin(poa_lookup["poa_code_2021"])
].copy()

print("sal_to_poa filtered:", sal_to_poa.shape)

# SAL -> SA2 via POA
sal_to_sa2 = sal_to_poa.merge(
    poa_to_sa2[["poa_code_2021", "sa2_code_2021", "overlap_area_pct"]],
    on="poa_code_2021",
    how="inner"
)

sal_to_sa2["overlap_area_pct"] = (
    sal_to_sa2["sal_poa_overlap_pct"].fillna(0) *
    sal_to_sa2["overlap_area_pct"].fillna(0) / 100
).round(2)

sal_to_sa2 = (
    sal_to_sa2.groupby(["sal_code_2021", "sa2_code_2021"], as_index=False)["overlap_area_pct"]
    .sum()
)

sal_to_sa2 = sal_to_sa2.merge(
    sal_lookup[["sal_code_2021", "sal_name_2021"]],
    on="sal_code_2021",
    how="left"
)

sal_to_sa2 = sal_to_sa2.merge(
    sa2_lookup[["sa2_code_2021", "sa2_name_2021"]],
    on="sa2_code_2021",
    how="left"
)

sal_to_sa2["confidence"] = confidence_from_pct(sal_to_sa2["overlap_area_pct"].fillna(0))
sal_to_sa2 = sal_to_sa2[
    ["sal_code_2021", "sal_name_2021", "sa2_code_2021", "sa2_name_2021", "overlap_area_pct", "confidence"]
].drop_duplicates().sort_values(["sal_code_2021", "overlap_area_pct"], ascending=[True, False])

sal_to_sa2.to_csv(OUT_DIR / "sal_to_sa2_vic.csv", index=False)

print("sal_to_sa2_vic:", sal_to_sa2.shape)
display(sal_to_sa2.head(10))


poa_to_sa2_vic: (1435, 6)


,poa_code_2021,postcode,sa2_code_2021,sa2_name_2021,overlap_area_pct,confidence
2,POA3000,3000,206041504,Melbourne CBD - North,39.16,low
3,POA3000,3000,206041505,Melbourne CBD - West,36.33,low
1,POA3000,3000,206041503,Melbourne CBD - East,22.84,low
0,POA3000,3000,206041117,Carlton,1.43,low
4,POA3000,3000,206041506,North Melbourne,0.24,low
5,POA3002,3002,206041119,East Melbourne,100.00,high
8,POA3003,3003,206041510,West Melbourne - Residential,92.86,high
7,POA3003,3003,206041506,North Melbourne,7.13,low
6,POA3003,3003,206041127,West Melbourne - Industrial,0.00,low
12,POA3004,3004,206051128,Albert Park,50.44,low


sal_to_poa filtered: (2964, 3)
sal_to_sa2_vic: (8829, 6)


,sal_code_2021,sal_name_2021,sa2_code_2021,sa2_name_2021,overlap_area_pct,confidence
1,SAL20001,Abbeyard,204031071,Myrtleford,97.25,high
0,SAL20001,Abbeyard,204031069,Bright - Mount Beauty,2.75,low
2,SAL20002,Abbotsford (Vic.),206071139,Abbotsford,100.00,high
3,SAL20002,Abbotsford (Vic.),206071145,Clifton Hill - Alphington,0.00,low
4,SAL20003,Aberfeldie,206031501,Essendon (West) - Aberfeldie,48.51,low
5,SAL20003,Aberfeldie,206031502,Essendon - East,45.63,low
6,SAL20003,Aberfeldie,210011230,Niddrie - Essendon West,5.86,low
9,SAL20004,Aberfeldy,205041095,Moe - Newborough,82.06,medium
7,SAL20004,Aberfeldy,205011077,Mount Baw Baw Region,10.55,low
10,SAL20004,Aberfeldy,205041098,Yallourn North - Glengarry,7.14,low
